In [58]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
from pathlib import Path
import numpy as np

import sys
sys.path.append('../scripts')
from data_utils import *


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
print("\n===================== vispils-v1.csv ====================\n")
load_csv_info('../../vispils/data/vispils-v1.csv')
print("\n===================== data_vispils.csv ====================\n")
load_csv_info('../../vispils/data/data_vispils.csv')


===================== vispils-v1.csv ====================

File: vispils-v1.csv
Size: 333.40 MB
Rows: 337,289
Columns: 88
Column Names: ['Iso SMILES', 'GC Pred.', 'aSMILES', 'afam1', 'afam_class', 'aCenterValence', 'aElectronegativity', 'aCenterSize', 'aSymmetry', 'aOxyAnions', 'aDipole', 'cfam', 'cfam1', 'cfam_class', 'cfamVolume', 'ctailVolume', 'cHeadVertex', 'cHeteroAtom', 'cFGs', 'cFGsGroup', 'aMolLogP', 'aMolMR', 'aMolWt', 'aNumHBA', 'aNumHBD', 'aRings', 'aVolume', 'aAsphericity', 'aEccentricity', 'aInertialShapeFactor', 'aPBF', 'aPMI1', 'aPMI2', 'aPMI3', 'aNPR1', 'aNPR2', 'aRadiusOfGyration', 'aSpherocityIndex', 'aLabuteASA', 'aTPSA', 'aAliCC', 'aAliHC', 'aAliRings', 'aAroCC', 'aAroHC', 'aAroRings', 'aNumNHOH', 'cMolLogP', 'cMolMR', 'cMolWt', 'cNumHBA', 'cNumHBD', 'cRings', 'cVolume', 'cAsphericity', 'cEccentricity', 'cInertialShapeFactor', 'cPBF', 'cPMI1', 'cPMI2', 'cPMI3', 'cNPR1', 'cNPR2', 'cRadiusOfGyration', 'cSpherocityIndex', 'cLabuteASA', 'cTPSA', 'cAliCC', 'cAliHC', 'c

(3.080571174621582,
 20252,
 12,
 ['Dataset',
  'Iso SMILES',
  'cSMILES',
  'aSMILES',
  'Pressure',
  'Temperature',
  'Log viscosity',
  'Viscosity',
  'cfam',
  'afam',
  'cvolume',
  'avolume'])

## vispils-v1.csv

This file includes both experimental and combinatorial ils
- number of exp. ILs: 1,959
- number of pred. ILs: 335,330

In [3]:
# load_csv_info('../../vispils/data/vispils-v1.csv')

In [4]:
df_ils = pd.read_csv('../../vispils/data/vispils-v1.csv')

In [65]:
df_ils = dfUtils(df_ils)


df_ils.data_summary(head=False, 
                    smiles_col= "Iso SMILES", temperature_col="Temperature")

print(df_ils['DataSource'].unique())
df_ils.groupby('DataSource').size()


====== data summary ======

Total data points (337289)
Unique IL SMILES (337274)
Unique temperatures (88): [298.15 298.   298.2  298.1  298.13 298.24 298.3  298.05]
Columns (88): ['Anion|cFGsGroup', 'Anion|cfam', 'Anion|cfam_class', 'Anion|cfam|cFGsGroup', 'DataSource', 'Dataset', 'GC Pred.', 'Iso SMILES', 'Log viscosity', 'Reliability', 'Temperature', 'Validated', 'Vm', 'aAliCC', 'aAliHC', 'aAliRings', 'aAroCC', 'aAroHC', 'aAroRings', 'aAsphericity', 'aCenterSize', 'aCenterValence', 'aDipole', 'aEccentricity', 'aElectronegativity']
['Pred.' 'Exp.']


DataSource
Exp.       1959
Pred.    335330
dtype: int64

In [66]:
df_ils_sanitized = df_ils.sanitize_old_df_cols(inplace=False)
df_ils_sanitized = dfUtils(df_ils_sanitized)
df_ils_sanitized.data_summary(head=False)

✓ Renaming 14 column(s):
  'Dataset' → 'reference'
  'Iso SMILES' → 'il_smiles'
  'cSMILES' → 'cation_smiles'
  'aSMILES' → 'anion_smiles'
  'Temperature' → 'temperature_k'
  'η (mPas)' → 'viscosity_mpas'
  'Log viscosity' → 'log_10_viscosity_mpas'
  'cfam' → 'cation_family_v0'
  'cfam1' → 'cation_family'
  'afam1' → 'anion_family'
  'cfam_class' → 'cation_class'
  'afam_class' → 'anion_class'
  'cFGs' → 'cation_functional_group'
  'cFGsGroup' → 'cation_functional_group_class'
ℹ️  Note: 5 column(s) not found (not renamed):
  - 'aFGsGroup'
  - 'Viscosity'
  - 'aFGs'
  - 'afam'
  - 'Pressure'
ℹ️  74 column(s) remain unchanged

====== data summary ======

Total data points (337289)
Unique IL SMILES (337274)
Unique temperatures (88): [298.15 298.   298.2  298.1  298.13 298.24 298.3  298.05]
Columns (88): ['Anion|cFGsGroup', 'Anion|cfam', 'Anion|cfam_class', 'Anion|cfam|cFGsGroup', 'DataSource', 'GC Pred.', 'Reliability', 'Validated', 'Vm', 'aAliCC', 'aAliHC', 'aAliRings', 'aAroCC', 'aAroHC

In [60]:
# Check chemical structure derived category columns

group_summary(df_ils_sanitized, 'cation_family')
group_summary(df_ils_sanitized, 'cation_class')
group_summary(df_ils_sanitized, 'anion_family')
group_summary(df_ils_sanitized, 'anion_class')
group_summary(df_ils_sanitized, 'cation_functional_group')
group_summary(df_ils_sanitized, 'anion_functional_group')


Group summary by cation_family:
number of unique cation_family: 9
----------------------------------------


cation_family
im       152325
mor        7224
n         82484
other       133
p         23178
pip       11739
py        37023
pyr       14755
s          8428
dtype: int64

----------------------------------------

Group summary by cation_class:
number of unique cation_class: 4
----------------------------------------


cation_class
Aromatic       189348
Non-ring       114090
Nonaromatic     33718
other             133
dtype: int64

----------------------------------------

Group summary by anion_family:
number of unique anion_family: 13
----------------------------------------


...

anion_class
[N-]      29218
[O-]     182598
cyc       25764
mln       68334
oph       31364
other        11
dtype: int64

----------------------------------------

Group summary by cation_functional_group:
number of unique cation_functional_group: 95
----------------------------------------


cation_functional_group
*#N                  14749
*#N|-OH                301
*#N|C-O-C              301
*#N|C-O-C|C=O          301
*#N|CN                 301
                     ...  
[pyr]1|C=O            2414
[s,p]Cyc5             1806
[s,p]Cyc5|Benzene      301
[s,p]Cyc6             1204
[s,p]Cyc6|Benzene      301
Length: 95, dtype: int64

----------------------------------------


ValueError: Column 'anion_functional_group' not found in dataframe.

In [68]:
group_summary(df_ils_sanitized, 'DataSource')
group_summary(df_ils_sanitized, 'reference')


Group summary by DataSource:
number of unique DataSource: 2
----------------------------------------


DataSource
Exp.       1959
Pred.    335330
dtype: int64

----------------------------------------

Group summary by reference:
number of unique reference: 6
----------------------------------------


reference
Exp.Non-VBM                    1186
Exp.VBM                         773
Expanded Exp.VBM               4419
Expanded Exp.VBM Rejected      1935
Pred.Non-VBM                 170351
Pred.VBM                     158625
dtype: int64

----------------------------------------


## data_vispils.csv

This is the raw experimental dataset

In [76]:
df_visc_exp = pd.read_csv('../../vispils/data/data_vispils.csv')

In [77]:
# Get a summary of the dataframe
df_visc_exp = dfUtils(df_visc_exp)
df_visc_exp.data_summary(head=False, smiles_col='Iso SMILES', temperature_col='Temperature')

# Santitize column names
df_visc_exp_sanitized = df_visc_exp.sanitize_old_df_cols(inplace=False)
df_visc_exp_sanitized = dfUtils(df_visc_exp_sanitized)
df_visc_exp_sanitized.data_summary(smiles_col='il_smiles', temperature_col='temperature_k',
                          head=False)



====== data summary ======

Total data points (20252)
Unique IL SMILES (2951)
Unique temperatures (12): [253.0, 253.15, 253.2, 258.15, 259.9, 262.61, 262.7, 263.1, 263.15, 263.42, 264.05, 264.25, 264.5, 264.9, 265.1, 265.65, 265.94, 266.6, 266.77, 267.3, 267.59, 268.0, 268.15, 268.41, 269.1]
Columns (12): Index(['Dataset', 'Iso SMILES', 'cSMILES', 'aSMILES', 'Pressure',
       'Temperature', 'Log viscosity', 'Viscosity', 'cfam', 'afam', 'cvolume',
       'avolume'],
      dtype='object')
✓ Renaming 10 column(s):
  'Dataset' → 'reference'
  'Iso SMILES' → 'il_smiles'
  'cSMILES' → 'cation_smiles'
  'aSMILES' → 'anion_smiles'
  'Pressure' → 'pressure_atm'
  'Temperature' → 'temperature_k'
  'Viscosity' → 'viscosity_mpas'
  'Log viscosity' → 'log_10_viscosity_mpas'
  'cfam' → 'cation_family_v0'
  'afam' → 'anion_family_v0'
ℹ️  Note: 9 column(s) not found (not renamed):
  - 'cFGsGroup'
  - 'η (mPas)'
  - 'cFGs'
  - 'cfam_class'
  - 'aFGsGroup'
  - 'afam_class'
  - 'aFGs'
  - 'afam1'
  - '

In [81]:
group_summary(df_visc_exp_sanitized, 'reference')


Group summary by reference:
number of unique reference: 3
----------------------------------------


reference
ILthermo     4565
Norway       4479
Poland      11208
dtype: int64

----------------------------------------


## Sample datasets for demonstration

In [81]:
# Sample combinatorial data points
df_ils_sample1 = df_ils_sanitized[df_ils_sanitized['DataSource'] == 'Exp.'].sample(20, random_state=42)
df_ils_sample2 = df_ils_sanitized[df_ils_sanitized['DataSource'] == 'Pred.'].sample(500, random_state=42)

df_ils_sample = pd.concat([df_ils_sample1, df_ils_sample2], ignore_index=True)

df_ils_sample = dfUtils(df_ils_sample)
df_ils_sample.data_summary(temp_col='temperature_k', head=False)


SAMPLE_COLS = ['DataSource', 
               'il_smiles', 'cation_smiles', 'anion_smiles',
               'temperature_k', 'log_10_viscosity_mpas', 'viscosity_mpas', 
               'cation_family', 'anion_family', 'cVolume',
               'aVolume', 'cation_class', 'anion_class']
df_ils_sample = df_ils_sample[SAMPLE_COLS]
df_ils_sample.head()



====== data summary ======

Total data points (520)
Unique IL SMILES (520)
Unique temperatures (88): [298.2  298.   298.15 298.1  298.13]
Columns (88): ['Anion|cFGsGroup', 'Anion|cfam', 'Anion|cfam_class', 'Anion|cfam|cFGsGroup', 'DataSource', 'GC Pred.', 'Reliability', 'Validated', 'Vm', 'aAliCC', 'aAliHC', 'aAliRings', 'aAroCC', 'aAroHC', 'aAroRings', 'aAsphericity', 'aCenterSize', 'aCenterValence', 'aDipole', 'aEccentricity', 'aElectronegativity', 'aInertialShapeFactor', 'aLabuteASA', 'aMolLogP', 'aMolMR']


,DataSource,il_smiles,cation_smiles,anion_smiles,temperature_k,log_10_viscosity_mpas,viscosity_mpas,cation_family,cation_family,anion_family,cVolume,aVolume,cation_class,anion_class
0,Exp.,CCNC(NCC)=[S+]CC.O=S(=O)([N-]S(=O)(=O)C(F)(F)F...,CCNC(NCC)=[S+]CC,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,298.20,1.770852,59.000000,turea,other,ntf,0.140264,0.155344,other,[N-]
1,Exp.,CCCCC[N+](CC)(CC)CC.N#C[N-]C#N,CCCCC[N+](CC)(CC)CC,N#C[N-]C#N,298.00,2.080000,120.226443,n,n,dca,0.169480,0.059040,Non-ring,[N-]
2,Exp.,CCCCCCSC(N(C)C)=[N+](C)C.O=S(=O)([N-]S(=O)(=O)...,CCCCCCSC(N(C)C)=[N+](C)C,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,298.00,1.930000,85.113804,thur,other,ntf,0.190488,0.155344,other,[N-]
3,Exp.,CCCCCCCC[n+]1ccccc1C.O=S(=O)([N-]S(=O)(=O)C(F)...,CCCCCCCC[n+]1ccccc1C,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,298.15,2.086360,122.000000,py,py,ntf,0.189744,0.155344,Aromatic,[N-]
4,Exp.,[NH3+]CCO.CC(O)C(=O)[O-],[NH3+]CCO,CC(O)C(=O)[O-],298.20,2.405005,254.099980,n,n,co,0.054520,0.071448,Non-ring,[O-]


In [30]:
# Sample 20 experimental data points from ILthermo

df_visc_sample = df_visc[df_visc['Dataset'] == 'ILthermo'].sample(20, random_state=42)
df_visc_sample = dfUtils(df_visc_sample)
df_visc_sample.data_summary(head=False)




====== data summary ======

Total data points (20)
Unique IL SMILES (19)
Unique temperatures (14): [308.64 303.15 333.15 305.63 353.15 298.15 343.15 373.15 323.15 298.35
 278.15 343.   328.15 318.1 ]
Columns (12): Index(['Dataset', 'Iso SMILES', 'cSMILES', 'aSMILES', 'Pressure',
       'Temperature', 'Log viscosity', 'Viscosity', 'cfam', 'afam', 'cvolume',
       'avolume'],
      dtype='object')


## Export Files

In [82]:
# Save sampled ils to path
csv_path = Path('../../vispils/data/vispils_il_combinatorial_sample520.csv')
df_ils_sample.to_csv(csv_path, index=False)
print(f"Saved sample experimental data to {csv_path.resolve()}")

Saved sample experimental data to /Users/huangziru/Documents/Repos/vispils-flask-app/vispils/data/vispils_il_combinatorial_sample520.csv


In [69]:
# # Save sampled experimental viscosity to path
# csv_path = Path('../../vispils/data/vispils_viscosity_exp_sample20.csv')
# df_visc_sample.to_csv(csv_path, index=False)
# print(f"Saved sample experimental data to {csv_path.resolve()}")

In [ ]:
# Save sanitized full experimental data to path
csv_path = Path('../../vispils/data/vispils_exp_data.csv')
df_sanitized.to_csv(csv_path, index=False)
print(f"Saved full experimental data to {csv_path.resolve()}")

Saved full experimental data to /Users/huangziru/Documents/Repos/vispils-flask-app/vispils/data/vispils_exp_data.csv
